# 🎙️ Qwen3-TTS 1.7B Studio - Interactive Web UI (Human Mode)
### Advanced Text-to-Speech: Voice Cloning, Custom Design & Natural Language Style Instructions
* **Models:** `Qwen/Qwen3-TTS-12Hz-1.7B-Base` & `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`
* **Features:** Zero-shot Voice Cloning, Natural Language Voice Styling, and Studio Loudness Mastering (-14 LUFS).
* **Mode:** **Interactive Gradio Web UI** (exposed via Cloudflare tunnel).


In [ ]:
# [Step 1/3] Environment & Dependencies Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

print(">>> [1/3] Installing Dependencies & System Tools...", flush=True)

# Install Cloudflared
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True, check=True
)

subprocess.run("apt-get update -qq && apt-get install -y -qq sox libsox-fmt-all > /dev/null 2>&1", shell=True)

pkgs = ["transformers==4.57.3", "accelerate>=1.12.0", "soundfile", "pyloudnorm", "gradio==5.29.0", "qwen-tts"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✓ Environment ready.", flush=True)


In [ ]:
# [Step 2/3] Load Qwen3-TTS Models & Mastering Engine
import os, time, warnings, logging, io, contextlib
import numpy as np, soundfile as sf, torch, pyloudnorm as pyln

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from qwen_tts import Qwen3TTSModel

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = True

print(">>> [2/3] Loading Qwen3-TTS Models...", flush=True)
t0 = time.time()
base_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
print(f"✓ Base model loaded in {time.time() - t0:.1f}s", flush=True)

t1 = time.time()
design_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
print(f"✓ VoiceDesign model loaded in {time.time() - t1:.1f}s", flush=True)

def master_audio(wav, sr, target_lufs=-14.0):
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.squeeze()
    wav = wav - np.mean(wav)
    try:
        meter = pyln.Meter(sr)
        current_lufs = meter.integrated_loudness(wav)
        if not np.isneginf(current_lufs) and not np.isnan(current_lufs):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                wav = pyln.normalize.loudness(wav, current_lufs, target_lufs)
    except Exception:
        pass
    peak = np.max(np.abs(wav))
    if peak > 0.95:
        wav = wav * (0.95 / peak)
    return wav


In [ ]:
# [Step 3/3] Launch Interactive Gradio Studio UI & Cloudflare Tunnel
import gradio as gr, subprocess, threading, re, tempfile

def tts_generate(text, mode, ref_audio, voice_prompt):
    if not text.strip():
        return None, "Please provide dialogue text."
    
    out_dir = "/kaggle/working/outputs"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"speech_{int(time.time())}.wav")
    
    if mode == "Voice Cloning (Audio Reference)":
        if not ref_audio:
            return None, "Please upload a reference audio file for voice cloning."
        wav, sr = base_model.generate_from_audio(text, ref_audio)
    else: # Voice Design
        prompt = voice_prompt if voice_prompt.strip() else "warm, natural conversational voice"
        wav, sr = design_model.generate_from_description(text, prompt)
    
    mastered = master_audio(wav, sr)
    sf.write(out_path, mastered, sr, subtype="PCM_16")
    return out_path, f"Rendered successfully ({len(mastered)/sr:.1f}s, {sr} Hz)"

with gr.Blocks(title="Qwen3-TTS Audio Studio") as demo:
    gr.Markdown("# 🎙️ Qwen3-TTS 1.7B Voice Studio")
    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(label="Dialogue Text (Hindi / English / Multilingual)", lines=4, placeholder="Type dialogue text with pacing marks (e.g. 'Ruk jao... Maine kaha ruk jao!')")
            mode_choice = gr.Radio(["Voice Design (Natural Description)", "Voice Cloning (Audio Reference)"], label="Synthesis Mode", value="Voice Design (Natural Description)")
            voice_desc = gr.Textbox(label="Voice Description (for Voice Design)", placeholder="e.g. A slow, chilling villain with a gravelly baritone voice and deep pauses.")
            ref_audio_input = gr.Audio(label="Reference Audio (for Voice Cloning)", type="filepath")
            gen_btn = gr.Button("Generate Speech", variant="primary")
        with gr.Column():
            audio_output = gr.Audio(label="Rendered & Mastered Audio (-14 LUFS)")
            status_output = gr.Textbox(label="Status")
            
    gen_btn.click(tts_generate, inputs=[text_input, mode_choice, ref_audio_input, voice_desc], outputs=[audio_output, status_output])

PORT = 7860
demo.queue()
threading.Thread(target=lambda: demo.launch(server_name="127.0.0.1", server_port=PORT, quiet=True), daemon=True).start()

# Cloudflare Tunnel
tunnel_proc = subprocess.Popen(f"/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
t_start = time.time()
while time.time() - t_start < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break

print("\n" + "=" * 80, flush=True)
if public_url:
    print(f"🌐 GRADIO LIVE WEB UI: {public_url}", flush=True)
    print("👉 Open the link above in your browser to generate speech interactively!", flush=True)
print("=" * 80 + "\n", flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 5 == 0:
        print(f"[Heartbeat] Qwen3-TTS studio is online ({counter} min).", flush=True)
